In [1]:
import sys
sys.path.append('..')

from typing import Literal, Optional
from pydantic import BaseModel, Field
from utils.jason_utils import (format_pydantic_schema_for_prompt)
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.prompts import render
from utils.router import pick_model
from utils.jason_utils import safe_parse_json
from utils.report_utils import save_events_to_excel
import logging
from pydantic import ValidationError

In [2]:
class CrisisEvent(BaseModel):
    district: Literal["Colombo", "Gampaha", "Kandy", "Kalutara", "Galle", "Kegalle", "Ratnapura", "Matara", "Nuwara Eliya"] = Field(..., description="The district affected by the crisis")
    flood_level_meters: float = Field(default=None, description="The flood level in meters")
    victim_count: int = Field(default=0, description="The number of victims")
    main_need: str = Field(..., description="The main need of the victims")
    status: Literal["Critical", "Warning", "Stable"] = Field(..., description="The current status of the situation")
    
schema_str = format_pydantic_schema_for_prompt(CrisisEvent)
    
model = pick_model('openai', 'general')
client = LLMClient('openai', model)

with open('..\\data\\News Feed.txt', 'r') as files:
    lines = files.readlines()

valid_events = []

for line in lines:
    prompt_text, spec = render(
        "json_extract.v1", 
        schema=schema_str, 
        text=line
    )
    
    response = client.json_chat(
        [
            {"role": "user", "content": prompt_text}
        ],
        temperature=0.0
    )
    
    log_llm_call('openai', model, 'json_extract', response['latency_ms'], response['usage'])
    
    json_str = response['text']
    
    try:
        event = CrisisEvent.model_validate_json(json_str)
        valid_events.append(event)
    except ValidationError as e:
        logging.warning(f"Validation failed: {e}")
        continue

if valid_events:
    # Pass the list of Pydantic objects directly
    save_events_to_excel(valid_events)
else:
    print("No valid events extracted. Report not created.")

flood_level_meters
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/float_type
flood_level_meters
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/float_type
flood_level_meters
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/float_type
flood_level_meters
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/float_type
flood_level_meters
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/float_type
flood_level_meters
  Input should be a valid number [ty

Successfully saved 10 events to ..\output\flood_report.xlsx


d:\rtyuik\FGUEWORLsQ\notebooks\..\utils\report_utils.py:60: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  updated_df = pd.concat([existing_df, new_df], ignore_index=True)
